# 🚀 MISSÃO AURORA SIGER — Ignition Zero: O Início da Aurora
**Relatório Operacional de Pré-Decolagem**

**Autor:** Francisco Wellington da Silva Rodrigues | **RM:** 566700

---

## 1. Carregamento e Organização da Telemetria (CSV)
Todos os dados são lidos diretamente do arquivo `telemetria_aurora.csv`, que contém 30 leituras de sensores em intervalos de 5 minutos.

In [2]:
import csv
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

CSV_PATH = 'telemetria_aurora.csv'

# ── Leitura com pandas ──
df = pd.read_csv(CSV_PATH, parse_dates=['timestamp'])
print(f'✅ CSV carregado: {len(df)} leituras | {df.shape[1]} colunas')
print(f'   Período: {df.timestamp.iloc[0]}  →  {df.timestamp.iloc[-1]}')
print()
df.head()

FileNotFoundError: [Errno 2] No such file or directory: ''

In [ ]:
# Estatísticas descritivas dos campos numéricos
cols_num = ['temperatura_interna','temperatura_externa',
            'energia_banco_A','energia_banco_B','energia_banco_C',
            'pressao_combustivel','pressao_oxidante']
df[cols_num].describe().round(2)

## 2. Visualização da Telemetria ao Longo do Tempo

In [ ]:
fig = plt.figure(figsize=(16, 12))
fig.patch.set_facecolor('#0d1b2a')
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

palette = {'A':'#f39c12','B':'#e67e22','C':'#d35400'}
tick_kw = dict(colors='#a0b0c0')
spine_color = '#2e4a6a'

def style_ax(ax, title):
    ax.set_facecolor('#0d2035')
    ax.set_title(title, color='white', fontweight='bold', fontsize=10, pad=8)
    ax.tick_params(**tick_kw)
    for s in ax.spines.values(): s.set_color(spine_color)
    ax.xaxis.label.set_color('#a0b0c0')
    ax.yaxis.label.set_color('#a0b0c0')
    ax.grid(color='#1e3a5f', linestyle='--', linewidth=0.5, alpha=0.7)
    ax.tick_params(axis='x', rotation=30, labelsize=7)

x = df['timestamp']

# 1 – Temperaturas
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(x, df['temperatura_interna'], color='#2ecc71', lw=2, label='Interna')
ax1.plot(x, df['temperatura_externa'], color='#3498db', lw=2, label='Externa')
ax1.axhline(35, color='#e74c3c', ls='--', lw=0.8, alpha=0.7, label='Lim. max int.')
ax1.axhline(15, color='#f39c12', ls='--', lw=0.8, alpha=0.7, label='Lim. min int.')
ax1.legend(fontsize=7, facecolor='#0d2035', labelcolor='white', framealpha=0.8)
ax1.set_ylabel('°C')
style_ax(ax1, 'Temperaturas (°C)')

# 2 – Energia bancos
ax2 = fig.add_subplot(gs[0, 1])
for banco, cor in palette.items():
    ax2.plot(x, df[f'energia_banco_{banco}'], color=cor, lw=2, label=f'Banco {banco}')
ax2.axhline(80, color='#e74c3c', ls='--', lw=0.8, alpha=0.7, label='Mínimo (80%)')
ax2.set_ylabel('%')
ax2.legend(fontsize=7, facecolor='#0d2035', labelcolor='white', framealpha=0.8)
style_ax(ax2, 'Energia por Banco (%)')

# 3 – Pressão tanques
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(x, df['pressao_combustivel'], color='#9b59b6', lw=2, label='Combustível')
ax3.plot(x, df['pressao_oxidante'],    color='#8e44ad', lw=2, ls='--', label='Oxidante')
ax3.axhline(340, color='#e74c3c', ls=':', lw=0.8, alpha=0.6, label='Max Comb.')
ax3.axhline(280, color='#f39c12', ls=':', lw=0.8, alpha=0.6, label='Min Comb.')
ax3.set_ylabel('bar')
ax3.legend(fontsize=7, facecolor='#0d2035', labelcolor='white', framealpha=0.8)
style_ax(ax3, 'Pressão dos Tanques (bar)')

# 4 – Integridade estrutural
ax4 = fig.add_subplot(gs[1, 1])
ax4.fill_between(x, df['integridade_estrutural'], color='#27ae60', alpha=0.6, label='Integridade')
ax4.set_ylim(-0.1, 1.5)
ax4.set_yticks([0, 1])
ax4.set_yticklabels(['FALHA (0)', 'OK (1)'], color='#a0b0c0')
ax4.legend(fontsize=7, facecolor='#0d2035', labelcolor='white', framealpha=0.8)
style_ax(ax4, 'Integridade Estrutural (0/1)')

# 5 – Módulos (mapa de calor de status ao longo do tempo)
ax5 = fig.add_subplot(gs[2, :])
modulos = ['modulo_propulsao','modulo_navegacao','modulo_comunicacao',
           'modulo_suporte_vida','modulo_energia']
labels  = ['Propulsão','Navegação','Comunicação','Suporte Vida','Energia']
mat = df[modulos].values.T.astype(float)
im = ax5.imshow(mat, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1,
                extent=[0, len(df)-1, -0.5, len(modulos)-0.5])
ax5.set_yticks(range(len(modulos)))
ax5.set_yticklabels(labels, color='#a0b0c0', fontsize=9)
ax5.set_xlabel('Leitura #', color='#a0b0c0')
ax5.set_facecolor('#0d2035')
ax5.set_title('Status dos Módulos Críticos ao Longo do Tempo (Verde=OK)',
              color='white', fontweight='bold', fontsize=10)
ax5.tick_params(colors='#a0b0c0')
for s in ax5.spines.values(): s.set_color(spine_color)

fig.suptitle('📡 Painel de Telemetria — Aurora Siger (30 leituras)',
             color='white', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('telemetria_painel.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1b2a')
plt.show()
print('✅ Painel salvo: telemetria_painel.png')

## 3. Algoritmo de Verificação

**Pseudocódigo:**
```
INÍCIO
  CARREGAR leituras de telemetria_aurora.csv
  snapshot ← última leitura do CSV
  alertas ← []

  SE temperatura_interna < 15 OU > 35        → alertas.add(TEMP_INT)
  SE temperatura_externa < -150 OU > 50      → alertas.add(TEMP_EXT)
  SE integridade_estrutural ≠ 1              → alertas.add(STRUCT)
  PARA banco em [A,B,C]:
    SE energia_banco < 80%                   → alertas.add(ENERGIA)
  SE pressao_combustivel ∉ [280, 340]        → alertas.add(PRESSAO_C)
  SE pressao_oxidante ∉ [260, 320]           → alertas.add(PRESSAO_O)
  PARA modulo em [propulsao...energia]:
    SE modulo == 0                           → alertas.add(MODULO)

  SE alertas == [] → PRONTO PARA DECOLAR
  SENÃO            → DECOLAGEM ABORTADA
FIM
```

## 4. Script Python — Implementação do Algoritmo com CSV

In [ ]:
LIMITES = {
    'temperatura_interna_min': 15.0,  'temperatura_interna_max': 35.0,
    'temperatura_externa_min': -150.0,'temperatura_externa_max': 50.0,
    'energia_minima':          80.0,
    'pressao_combustivel_min': 280.0, 'pressao_combustivel_max': 340.0,
    'pressao_oxidante_min':    260.0, 'pressao_oxidante_max':    320.0,
}

def verificar_snapshot(row, lim):
    alertas = []
    if not (lim['temperatura_interna_min'] <= row['temperatura_interna'] <= lim['temperatura_interna_max']):
        alertas.append(f"[TEMP_INT] {row['temperatura_interna']}°C fora da faixa")
    if not (lim['temperatura_externa_min'] <= row['temperatura_externa'] <= lim['temperatura_externa_max']):
        alertas.append(f"[TEMP_EXT] {row['temperatura_externa']}°C fora da faixa")
    if row['integridade_estrutural'] != 1:
        alertas.append('[STRUCT] Integridade comprometida')
    for b in ['A','B','C']:
        v = row[f'energia_banco_{b}']
        if v < lim['energia_minima']:
            alertas.append(f'[ENERGIA] Banco {b} = {v}% abaixo do mínimo')
    if not (lim['pressao_combustivel_min'] <= row['pressao_combustivel'] <= lim['pressao_combustivel_max']):
        alertas.append(f"[PRESSAO_C] {row['pressao_combustivel']} bar anormal")
    if not (lim['pressao_oxidante_min'] <= row['pressao_oxidante'] <= lim['pressao_oxidante_max']):
        alertas.append(f"[PRESSAO_O] {row['pressao_oxidante']} bar anormal")
    for m in ['propulsao','navegacao','comunicacao','suporte_vida','energia']:
        if not row[f'modulo_{m}']:
            alertas.append(f'[MODULO] {m.upper()} OFFLINE')
    return len(alertas) == 0, alertas

# Varredura histórica
anomalias = []
for _, row in df.iterrows():
    ok, als = verificar_snapshot(row, LIMITES)
    if not ok:
        anomalias.append({'timestamp': row['timestamp'], 'alertas': als})

print(f'🔍 Varredura: {len(df)} leituras analisadas')
if anomalias:
    print(f'  ⚠️  {len(anomalias)} leitura(s) com anomalia encontrada(s)')
    for a in anomalias:
        print(f'  [{a["timestamp"]}]')
        for al in a['alertas']:
            print(f'    → {al}')
else:
    print('  ✅ Nenhuma anomalia em todo o histórico.')

# Verificação do snapshot final
snap = df.iloc[-1]
aprovado, alertas = verificar_snapshot(snap, LIMITES)
print(f'\n🔎 Verificação final — snapshot {snap["timestamp"]}')
if alertas:
    for a in alertas: print(f'  → {a}')
else:
    print('  ✅ Todos os parâmetros dentro das faixas.')

print('\n' + '─'*55)
print('🚀  RESULTADO: ✅ PRONTO PARA DECOLAR' if aprovado else '🛑  RESULTADO: ❌ DECOLAGEM ABORTADA')
print('─'*55)

## 5. Análise Energética

In [ ]:
CAPACIDADE_TOTAL = 4_500.0
CONSUMO_DEC      = 380.0
FATOR_PERDA      = 0.07

snap = df.iloc[-1]
carga_media  = (snap['energia_banco_A'] + snap['energia_banco_B'] + snap['energia_banco_C']) / 3
disponivel   = CAPACIDADE_TOTAL * (carga_media / 100)
perdas       = disponivel * FATOR_PERDA
util         = disponivel - perdas
autonomia_kw = util - CONSUMO_DEC
autonomia_pc = (autonomia_kw / CAPACIDADE_TOTAL) * 100

print('⚡ ANÁLISE ENERGÉTICA — AURORA SIGER')
print('─'*50)
print(f'  Capacidade Total         : {CAPACIDADE_TOTAL:>10,.1f} kWh')
print(f'  Carga Média (A+B+C)/3    : {carga_media:>10.2f} %')
print(f'  Energia Disponível       : {disponivel:>10,.1f} kWh')
print(f'  Perdas Estimadas (7%)    : {perdas:>10,.1f} kWh')
print(f'  Energia Útil             : {util:>10,.1f} kWh')
print(f'  Consumo na Decolagem     : {CONSUMO_DEC:>10,.1f} kWh')
print(f'  Autonomia Pós-Decolagem  : {autonomia_kw:>10,.1f} kWh  ({autonomia_pc:.1f}%)')
print('─'*50)

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1b2a')

# Evolução da energia ao longo do tempo
ax = axes[0]
ax.set_facecolor('#0d2035')
ax.plot(df['timestamp'], df['energia_banco_A'], '#f39c12', lw=2, label='Banco A')
ax.plot(df['timestamp'], df['energia_banco_B'], '#e67e22', lw=2, label='Banco B')
ax.plot(df['timestamp'], df['energia_banco_C'], '#d35400', lw=2, label='Banco C')
media_serie = (df['energia_banco_A'] + df['energia_banco_B'] + df['energia_banco_C'])/3
ax.plot(df['timestamp'], media_serie, 'white', lw=2.5, ls='--', label='Média')
ax.axhline(80, color='#e74c3c', ls='--', lw=1, alpha=0.8, label='Mínimo')
ax.set_title('Evolução dos Bancos de Energia', color='white', fontweight='bold')
ax.set_ylabel('%', color='#a0b0c0')
ax.tick_params(colors='#a0b0c0')
ax.tick_params(axis='x', rotation=30, labelsize=7)
for s in ax.spines.values(): s.set_color('#2e4a6a')
ax.grid(color='#1e3a5f', linestyle='--', linewidth=0.5, alpha=0.7)
ax.legend(fontsize=8, facecolor='#0d2035', labelcolor='white', framealpha=0.8)

# Distribuição energética (barras)
ax2 = axes[1]
ax2.set_facecolor('#0d2035')
cats   = ['Disponível', 'Perdas', 'Consumo\nDecolagem', 'Autonomia\nRestante']
vals   = [disponivel, perdas, CONSUMO_DEC, autonomia_kw]
cores  = ['#2ecc71', '#e74c3c', '#f39c12', '#3498db']
bars   = ax2.bar(cats, vals, color=cores, edgecolor='white', alpha=0.9, width=0.6)
for b, v in zip(bars, vals):
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+30,
             f'{v:,.0f}\nkWh', ha='center', va='bottom',
             color='white', fontweight='bold', fontsize=8)
ax2.set_title('Distribuição Energética', color='white', fontweight='bold')
ax2.set_ylabel('kWh', color='#a0b0c0')
ax2.tick_params(colors='#a0b0c0')
for s in ax2.spines.values(): s.set_color('#2e4a6a')
ax2.grid(color='#1e3a5f', linestyle='--', linewidth=0.5, alpha=0.7)

plt.tight_layout()
plt.savefig('analise_energetica.png', dpi=150, bbox_inches='tight', facecolor='#0d1b2a')
plt.show()
print('✅ Gráfico salvo: analise_energetica.png')

## 6. Análise Assistida por IA

Os dados do CSV foram submetidos à IA para classificação, identificação de anomalias e sugestões de risco.

### Classificação dos Dados

| Parâmetro | Média | Tendência | Status |
|---|---|---|---|
| Temperatura Interna | 25,8 °C | ↑ Crescente | ✅ Normal |
| Temperatura Externa | -65,8 °C | ↓ Decrescente | ✅ Normal |
| Integridade Estrutural | 1 (OK) | Estável | ✅ Normal |
| Energia Banco A | 88,8% | ↓ Drenagem gradual | ✅ Normal |
| Energia Banco B | 86,8% | ↓ Drenagem gradual | ✅ Normal |
| Energia Banco C | 91,0% | ↓ Drenagem gradual | ✅ Normal |
| Pressão Combustível | 317,2 bar | ↑ Crescente | ✅ Normal |
| Pressão Oxidante | 301,2 bar | ↑ Crescente | ✅ Normal |
| Módulos (5/5) | OPERACIONAL | Estável | ✅ Normal |

### Pontos de Atenção
- **Drenagem dos bancos:** queda média de ~0,3%/leitura. Se mantida, Banco B atingirá 80% em ~8 leituras (~40 min)
- **Temperatura interna crescente:** tendência de +0,31°C/leitura. Requer monitoramento se a tendência continuar
- **Pressão dos tanques em elevação:** dentro dos limites, mas aproximando-se dos 326 bar no combustível

### Sugestões de Risco
| Nível | Risco | Recomendação |
|---|---|---|
| 🟡 Baixo | Drenagem gradual dos bancos | Iniciar decolagem dentro da próxima janela |
| 🟡 Baixo | Temperatura interna em ascensão | Verificar sistema de climatização |
| 🟢 Negligível | Pressão dos tanques crescente | Manter monitoramento contínuo |

## 7. Reflexão Crítica

### Ética e Responsabilidade
A exploração espacial contemporânea exige que cada sistema computacional seja projetado com responsabilidade ética. Decisões automatizadas — como o algoritmo desta atividade — precisam ser transparentes, auditáveis e revisáveis por equipes humanas. A IA deve ser tratada como ferramenta de apoio, nunca como árbitro final.

### Impacto Social da Exploração Espacial
A Nova Corrida Espacial redefine fronteiras econômicas, políticas e sociais. Tecnologias como GPS e internet por satélite transformam a vida na Terra. Contudo, é fundamental que marcos regulatórios internacionais garantam distribuição justa dos benefícios para toda a humanidade.

### Sustentabilidade Tecnológica
A análise de drenagem dos bancos de energia — evidenciada pelo CSV — ilustra como a eficiência energética é crítica. Cada kWh economizado pode ser a diferença entre o sucesso e o aborto da missão, princípio que deve guiar qualquer sistema tecnológico contemporâneo.

In [ ]:
# Resumo final
print('='*55)
print('  RESUMO FINAL — MISSÃO AURORA SIGER')
print('='*55)
print(f'  Leituras CSV analisadas  : {len(df)}')
print(f'  Anomalias históricas     : {len(anomalias)}')
print(f'  Energia útil disponível  : {util:,.1f} kWh')
print(f'  Autonomia pós-decolagem  : {autonomia_pc:.1f}%')
print('-'*55)
print('🚀  AURORA SIGER: AUTORIZADA PARA DECOLAGEM' if aprovado else '🛑  DECOLAGEM ABORTADA')
print('='*55)